# Bagging：并行训练、投票决策
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：Bagging.py | 核心功能：Bootstrap 采样、并行集成、方差降低

## 概述

Bagging（Bootstrap Aggregating）是最基础的集成方法：通过 Bootstrap（有放回抽样）生成多个训练子集，在每个子集上训练一个基模型，最终取平均（回归）或投票（分类）。核心价值：**降低方差**，不改变偏差。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
# --- 导入库 ---
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

## 数学原理

### 1. Bootstrap 采样

Bagging 的核心是 **Bootstrap 自助采样**：从大小为 $n$ 的训练集 $D$ 中，有放回地随机抽取 $n$ 个样本组成子集 $D_b$。

每个样本在一次 Bootstrap 采样中**未被选中**的概率为：

$$\left(1 - \frac{1}{n}\right)^n \approx e^{-1} \approx 0.368$$

即每个基学习器平均只使用约 $63.2\%$ 的训练样本，剩余 $36.8\%$ 称为 **袋外样本（Out-of-Bag, OOB）**。

### 2. 方差缩减原理

设基学习器的预测方差为 $\text{Var}(f)$，预测之间的相关系数为 $\rho$，则 $B$ 个基学习器平均后的方差为：

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B}f_b\right) = \rho \cdot \text{Var}(f) + \frac{1-\rho}{B}\cdot \text{Var}(f)$$

- $\rho = 0$（完全独立）：方差缩小 $B$ 倍
- $\rho = 1$（完全相同）：无方差缩减
- 通过**随机性**（Bootstrap 采样 + 特征子采样）降低 $\rho$

### 3. 分类：多数投票

分类任务中，Bagging 的最终预测为：

$$\hat{y} = \arg\max_c \sum_{b=1}^{B} \mathbb{I}[f_b(x) = c]$$

其中 $\mathbb{I}[\cdot]$ 是指示函数。代码中 `BaggingClassifier` 默认使用硬投票。

### 4. 回归：均值聚合

回归任务中，最终预测为所有基学习器的简单平均：

$$\hat{y} = \frac{1}{B}\sum_{b=1}^{B} f_b(x)$$

### 5. OOB 误差估计

OOB 样本提供了一种**无需额外验证集**的泛化误差估计：

$$\text{OOB Error} = \frac{1}{n}\sum_{i=1}^{n} \mathbb{I}\left[y_i \neq \text{mode}\{f_b(x_i) : i \notin D_b\}\right]$$

对每个样本 $x_i$，只用那些**未包含** $x_i$ 的基学习器进行预测，取多数投票。

代码中 `oob_score=True` 计算此指标。

### 6. Bagging vs Pasting

| 采样方式 | 有放回 | 特点 |
|----------|--------|------|
| Bagging | 是 | 基学习器有差异，可用 OOB 评估 |
| Pasting | 否 | 基学习器更独立，但无 OOB |

代码中 `bootstrap=True` 为 Bagging，`bootstrap=False` 为 Pasting。

### 7. 随机子空间（特征采样）

代码中 `max_features=0.8` 实现**随机子空间法**：每个基学习器只使用 $80\%$ 的特征。结合样本采样，形成 **Random Patches** 方法，进一步降低 $\rho$。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `n_estimators=50` | 集成 $B=50$ 个基学习器 |
| `max_samples=0.8` | 每次 Bootstrap 采样 $0.8n$ 个样本 |
| `max_features=0.8` | 每次随机选取 $80\%$ 特征（随机子空间） |
| `bootstrap=True` | 有放回采样（Bagging） |
| `oob_score=True` | 计算袋外误差估计泛化性能 |
| `BaggingRegressor` | 回归聚合：$\hat{y} = \frac{1}{B}\sum f_b(x)$ |
| `BaggingClassifier` | 分类聚合：$\hat{y} = \arg\max_c \sum \mathbb{I}[f_b(x)=c]$ |

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. 单个决策树（基准）

运行 2. 单个决策树（基准） 的代码，观察算法在该环节的行为。

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("=== 单个决策树 ===")
print(f"测试准确率: {dt.score(X_test, y_test):.4f}")

### 3. Bagging 集成

运行 3. Bagging 集成 的代码，观察算法在该环节的行为。

In [ ]:
# n_estimators: 基学习器数量
# max_samples: 每个基学习器使用的样本比例
# max_features: 每个基学习器使用的特征比例
# bootstrap: 是否有放回采样（True=Bagging, False=Pasting）
# bootstrap_features: 是否对特征也做有放回采样

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
# --- 赋值/配置 ---
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)
bag.fit(X_train, y_train)

print(f"\n=== Bagging 集成 ===")
print(f"基学习器数量: {bag.n_estimators}")
print(f"训练准确率: {bag.score(X_train, y_train):.4f}")
print(f"测试准确率: {bag.score(X_test, y_test):.4f}")

### 4. OOB 评分

运行 4. OOB 评分 的代码，观察算法在该环节的行为。

In [ ]:
bag_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    oob_score=True,
    random_state=42,
# --- 继续 ---
)
bag_oob.fit(X_train, y_train)
print(f"OOB 准确率: {bag_oob.oob_score_:.4f}")

### 5. 参数对比

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== n_estimators 对比 ===")
for n in [5, 10, 25, 50, 100, 200]:
    b = BaggingClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    b.fit(X_train, y_train)
    print(f"  n={n:>3}: 测试准确率={b.score(X_test, y_test):.4f}")

print("\n=== max_samples 对比 ===")
for ms in [0.3, 0.5, 0.7, 0.8, 1.0]:
    b = BaggingClassifier(n_estimators=50, max_samples=ms, random_state=42)
    b.fit(X_train, y_train)
    print(f"  max_samples={ms}: 测试准确率={b.score(X_test, y_test):.4f}")

### 6. bootstrap vs pasting

运行 6. bootstrap vs pasting 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== bootstrap vs pasting ===")
for bootstrap in [True, False]:
    b = BaggingClassifier(n_estimators=50, bootstrap=bootstrap, random_state=42)
    b.fit(X_train, y_train)
    name = "Bagging" if bootstrap else "Pasting"
# --- 输出结果 ---
    print(f"  {name:>10}: 测试准确率={b.score(X_test, y_test):.4f}")

### 7. 特征子采样

运行 7. 特征子采样 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 特征子采样 ===")
for mf in [0.3, 0.5, 0.7, 1.0]:
    b = BaggingClassifier(n_estimators=50, max_features=mf, random_state=42)
    b.fit(X_train, y_train)
    print(f"  max_features={mf}: 测试准确率={b.score(X_test, y_test):.4f}")

### 8. Bagging 回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== Bagging 回归 ===")
from sklearn.datasets import make_regression
X_r, y_r = make_regression(n_samples=300, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

dt_r = DecisionTreeRegressor(random_state=42).fit(X_tr, y_tr)
bag_r = BaggingRegressor(n_estimators=50, random_state=42).fit(X_tr, y_tr)
print(f"决策树 R²: {dt_r.score(X_te, y_te):.4f}")
print(f"Bagging R²: {bag_r.score(X_te, y_te):.4f}")

print("\n=== Bagging 要点 ===")
print("- 核心：有放回采样 + 独立训练 + 投票/平均")
print("- 降低方差（适合高方差低偏差的基学习器如决策树）")
print("- OOB 评分：每个基学习器约用 63.2% 数据，剩余 36.8% 作 OOB 评估")
print("- 与随机森林的区别：随机森林额外做了特征随机选择")

## 关键代码解释

```python
from sklearn.ensemble import BaggingClassifier
bag = BaggingClassifier(n_estimators=50, max_samples=0.8, max_features=0.8)
bag.fit(X_train, y_train)
```

`max_samples` 和 `max_features` 控制每个基模型看到的数据和特征比例。

## 注意事项

1. 对高偏差模型（如浅层决策树）效果有限
2. 对高方差模型（如深层决策树）效果显著
3. 各基模型独立训练，可完全并行

## 总结与延伸

以上代码展示了 **Bagging** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **随机森林** = Bagging + 随机特征选择
- **Pasting**：不放回抽样的 Bagging
- **Subagging**：只用部分样本的 Bagging
